In [ ]:
import pandas as pd
import numpy as np
import random
import re
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


# ============================================================
# Paths / settings
# ============================================================

csv_path = "dense10_image_index_with_descriptions.csv"
output_path = "tangram_captioned.csv"

random_seed = 123
random.seed(random_seed)
np.random.seed(random_seed)

MAX_SWAP_COLUMNS = 3


# ============================================================
# Load
# ============================================================

df = pd.read_csv(csv_path)

TEXT_COLUMNS = [
    "photo_description",
    "part_description",
    "part_description_shape",
]

for col in TEXT_COLUMNS:
    if col not in df.columns:
        raise ValueError(f"Missing expected column: {col}")
    df[col] = df[col].fillna("").astype(str)


# ============================================================
# Similarity helpers
# ============================================================

# ============================================================
# Whole-annotation equivalence / bad swap filters
# ============================================================

STOPWORDS_FOR_OBJECT_MATCH = {
    "a", "an", "the", "of", "in", "on", "with", "and", "or", "to", "from",
    "at", "by", "for", "as", "like", "looks", "look", "image", "shows",
    "shape", "this", "that"
}

# Synonym / near-equivalence groups for this dataset.
# You can expand this as you notice more bad swaps.
EQUIVALENCE_GROUPS = [
    {"person", "human", "man", "woman", "girl", "boy", "lady", "dude", "male", "female"},
    {"house", "home"},
    {"dancer", "dancing", "dance"},
    {"prayer", "praying", "pray"},
    {"bird", "flying"},
    {"chair", "armchair"},
    {"snake"},
    {"dog", "puppy"},
    {"cat", "kitten"},
]

# Convert to lookup: token -> equivalence group id
EQUIV_LOOKUP = {}
for group_id, group in enumerate(EQUIVALENCE_GROUPS):
    for word in group:
        EQUIV_LOOKUP[word] = group_id


def normalize_for_object_match(text):
    text = str(text).lower().strip()

    # Remove the fixed caption shell if present.
    text = re.sub(r"^the image shows\s+", "", text)
    text = text.rstrip(".")

    # Normalize articles.
    text = re.sub(r"^(a|an|the)\s+", "", text)

    # Very light normalization for common variants.
    text = text.replace("arm chair", "armchair")

    return text.strip()


def object_tokens(text):
    """
    Extract content tokens from a whole-object phrase or photo_description.
    """
    text = normalize_for_object_match(text)
    tokens = re.findall(r"\b\w+\b", text.lower())

    cleaned = []
    for tok in tokens:
        if tok in STOPWORDS_FOR_OBJECT_MATCH:
            continue

        # crude lemmatization for simple plurals/gerunds
        if tok.endswith("ies") and len(tok) > 4:
            tok = tok[:-3] + "y"
        elif tok.endswith("ing") and len(tok) > 5:
            # dancing -> dance-ish; keep both original-ish and stem useful below
            tok = tok[:-3]
        elif tok.endswith("s") and not tok.endswith(("ss", "us")) and len(tok) > 3:
            tok = tok[:-1]

        cleaned.append(tok)

    return set(cleaned)


def equivalent_token_ids(tokens):
    """
    Map tokens into synonym/equivalence ids.
    Tokens not in a group get their own lexical id.
    """
    ids = set()

    for tok in tokens:
        if tok in EQUIV_LOOKUP:
            ids.add(f"GROUP_{EQUIV_LOOKUP[tok]}")
        else:
            ids.add(f"TOKEN_{tok}")

    return ids


def is_bad_whole_caption_swap(original_text, candidate_text):
    """
    Returns True when the candidate is too close to the original.

    Blocks:
      chair -> beach chair
      snake -> face of snake
      arm chair -> chair
      person hugging a pillow -> a person
      bird -> flying bird
      house -> home
      human -> person/man/woman/girl/boy
      dancer -> dancing human/man/woman
      prayer -> person praying
    """
    original_norm = normalize_for_object_match(original_text)
    candidate_norm = normalize_for_object_match(candidate_text)

    if not original_norm or not candidate_norm:
        return True

    if original_norm == candidate_norm:
        return True

    orig_tokens = object_tokens(original_norm)
    cand_tokens = object_tokens(candidate_norm)

    if not orig_tokens or not cand_tokens:
        return True

    # Exact content-word overlap blocks chair -> beach chair, bird -> flying bird, etc.
    if orig_tokens & cand_tokens:
        return True

    orig_equiv = equivalent_token_ids(orig_tokens)
    cand_equiv = equivalent_token_ids(cand_tokens)

    # Synonym / near-equivalence overlap blocks house/home, human/person, lady/woman, etc.
    if orig_equiv & cand_equiv:
        return True

    # Phrase containment after normalization.
    # Blocks cases like "man sitting" -> "man sitting on surfboard".
    if original_norm in candidate_norm or candidate_norm in original_norm:
        return True

    return False

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())


def tokenize(text):
    return set(re.findall(r"\b\w+\b", str(text).lower()))


def jaccard_similarity(a, b):
    a_tokens = tokenize(a)
    b_tokens = tokenize(b)

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def valid_other_caption(df, row_idx, candidate_idx, text_col):
    original = normalize_text(df.at[row_idx, text_col])
    candidate = normalize_text(df.at[candidate_idx, text_col])

    if row_idx == candidate_idx:
        return False

    if not candidate:
        return False

    if candidate == original:
        return False

    if text_col == "photo_description":
        if is_bad_whole_caption_swap(original, candidate):
            return False

    if text_col in {"part_description", "part_description_shape"}:
        original_whole = df.at[row_idx, "whole_annotation"]
        candidate_whole = df.at[candidate_idx, "whole_annotation"]

        if is_bad_whole_caption_swap(original_whole, candidate_whole):
            return False

    return True


def get_most_least_valid(sim_scores, df, row_idx, text_col):
    """
    Finds most/least similar captions while excluding:
    - self
    - candidates with exactly the same caption text
    - empty candidates
    """
    valid_indices = [
        cand_idx
        for cand_idx in df.index
        if valid_other_caption(df, row_idx, cand_idx, text_col)
    ]

    if not valid_indices:
        return None, np.nan, None, np.nan

    valid_scores = [(cand_idx, sim_scores[cand_idx]) for cand_idx in valid_indices]

    most_idx, most_score = max(valid_scores, key=lambda x: x[1])
    least_idx, least_score = min(valid_scores, key=lambda x: x[1])

    return int(most_idx), float(most_score), int(least_idx), float(least_score)


def add_random_caption_column(df, text_col, prefix):
    """
    Adds a random caption column.

    Tries to avoid reusing the same source row when possible, but always enforces:
      random_caption != original_caption
    """
    output_col = f"random_{prefix}"
    output_idx_col = f"random_{prefix}_source_idx"

    df[output_col] = ""
    df[output_idx_col] = np.nan

    all_indices = list(df.index)
    unused_candidates = all_indices.copy()
    random.shuffle(unused_candidates)

    for idx in df.index:
        # First try unused candidates so there are fewer repeats.
        valid_unused = [
            cand for cand in unused_candidates
            if valid_other_caption(df, idx, cand, text_col)
        ]

        if valid_unused:
            chosen_idx = valid_unused[0]
            unused_candidates.remove(chosen_idx)
        else:
            # Fallback: allow reuse if needed.
            valid_any = [
                cand for cand in all_indices
                if valid_other_caption(df, idx, cand, text_col)
            ]

            if not valid_any:
                chosen_idx = None
            else:
                chosen_idx = random.choice(valid_any)

        if chosen_idx is not None:
            df.at[idx, output_col] = df.at[chosen_idx, text_col]
            df.at[idx, output_idx_col] = chosen_idx

    return df


def add_similarity_swaps_for_column(df, text_col, prefix, sbert_model=None):
    """
    Adds random, Jaccard, TF-IDF, and SBERT most/least similar captions
    for one text column.
    """
    print(f"\nProcessing swaps for column: {text_col}")

    # -------------------------
    # Random caption
    # -------------------------

    df = add_random_caption_column(df, text_col, prefix)

    # -------------------------
    # Jaccard
    # -------------------------

    df[f"most_similar_{prefix}_jaccard"] = ""
    df[f"most_similar_{prefix}_jaccard_score"] = np.nan
    df[f"most_similar_{prefix}_jaccard_source_idx"] = np.nan

    df[f"least_similar_{prefix}_jaccard"] = ""
    df[f"least_similar_{prefix}_jaccard_score"] = np.nan
    df[f"least_similar_{prefix}_jaccard_source_idx"] = np.nan

    for i in df.index:
        scores = np.array([
            jaccard_similarity(df.at[i, text_col], other_text)
            for other_text in df[text_col]
        ])

        most_idx, most_score, least_idx, least_score = get_most_least_valid(
            scores, df, i, text_col
        )

        if most_idx is not None:
            df.at[i, f"most_similar_{prefix}_jaccard"] = df.at[most_idx, text_col]
            df.at[i, f"most_similar_{prefix}_jaccard_score"] = most_score
            df.at[i, f"most_similar_{prefix}_jaccard_source_idx"] = most_idx

        if least_idx is not None:
            df.at[i, f"least_similar_{prefix}_jaccard"] = df.at[least_idx, text_col]
            df.at[i, f"least_similar_{prefix}_jaccard_score"] = least_score
            df.at[i, f"least_similar_{prefix}_jaccard_source_idx"] = least_idx

    # -------------------------
    # TF-IDF cosine similarity
    # -------------------------

    df[f"most_similar_{prefix}_tfidf"] = ""
    df[f"most_similar_{prefix}_tfidf_score"] = np.nan
    df[f"most_similar_{prefix}_tfidf_source_idx"] = np.nan

    df[f"least_similar_{prefix}_tfidf"] = ""
    df[f"least_similar_{prefix}_tfidf_score"] = np.nan
    df[f"least_similar_{prefix}_tfidf_source_idx"] = np.nan

    # If the column is too empty, TfidfVectorizer can fail.
    nonempty_count = sum(bool(normalize_text(x)) for x in df[text_col])

    if nonempty_count > 1:
        vectorizer = TfidfVectorizer(stop_words="english")
        tfidf_matrix = vectorizer.fit_transform(df[text_col])
        tfidf_sim = cosine_similarity(tfidf_matrix)

        for i in df.index:
            most_idx, most_score, least_idx, least_score = get_most_least_valid(
                tfidf_sim[i], df, i, text_col
            )

            if most_idx is not None:
                df.at[i, f"most_similar_{prefix}_tfidf"] = df.at[most_idx, text_col]
                df.at[i, f"most_similar_{prefix}_tfidf_score"] = most_score
                df.at[i, f"most_similar_{prefix}_tfidf_source_idx"] = most_idx

            if least_idx is not None:
                df.at[i, f"least_similar_{prefix}_tfidf"] = df.at[least_idx, text_col]
                df.at[i, f"least_similar_{prefix}_tfidf_score"] = least_score
                df.at[i, f"least_similar_{prefix}_tfidf_source_idx"] = least_idx

    # -------------------------
    # Sentence-BERT similarity
    # -------------------------

    df[f"most_similar_{prefix}_sbert"] = ""
    df[f"most_similar_{prefix}_sbert_score"] = np.nan
    df[f"most_similar_{prefix}_sbert_source_idx"] = np.nan

    df[f"least_similar_{prefix}_sbert"] = ""
    df[f"least_similar_{prefix}_sbert_score"] = np.nan
    df[f"least_similar_{prefix}_sbert_source_idx"] = np.nan

    if sbert_model is not None and nonempty_count > 1:
        embeddings = sbert_model.encode(
            df[text_col].tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        sbert_sim = cosine_similarity(embeddings)

        for i in df.index:
            most_idx, most_score, least_idx, least_score = get_most_least_valid(
                sbert_sim[i], df, i, text_col
            )

            if most_idx is not None:
                df.at[i, f"most_similar_{prefix}_sbert"] = df.at[most_idx, text_col]
                df.at[i, f"most_similar_{prefix}_sbert_score"] = most_score
                df.at[i, f"most_similar_{prefix}_sbert_source_idx"] = most_idx

            if least_idx is not None:
                df.at[i, f"least_similar_{prefix}_sbert"] = df.at[least_idx, text_col]
                df.at[i, f"least_similar_{prefix}_sbert_score"] = least_score
                df.at[i, f"least_similar_{prefix}_sbert_source_idx"] = least_idx

    return df


# ============================================================
# Within-image shape swap helpers
# ============================================================

PIECE_TO_SHAPE = {
    "1": "square",
    "2": "parallelogram",
    "3": "small triangle",
    "4": "small triangle",
    "5": "medium triangle",
    "6": "large triangle",
    "7": "large triangle",
}


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def parse_json_cell(cell):
    if pd.isna(cell) or not str(cell).strip():
        return {}

    try:
        return json.loads(cell)
    except json.JSONDecodeError:
        return {}


def parse_bool_cell(x):
    if isinstance(x, bool):
        return x

    if pd.isna(x):
        return False

    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def without_leading_article(phrase):
    phrase = clean_text(phrase)
    words = phrase.split()

    if words and words[0].lower() in {"a", "an", "the"}:
        return " ".join(words[1:])

    return phrase


def parse_part_annotations_order(part_annotations):
    """
    Parses:
      "6: barrel; 4: head; 2: rump"

    Returns:
      [("6", "barrel"), ("4", "head"), ("2", "rump")]
    """
    part_annotations = clean_text(part_annotations)

    if not part_annotations:
        return []

    parsed = []

    for item in part_annotations.split(";"):
        item = item.strip()

        if not item or ":" not in item:
            continue

        piece_id, label = item.split(":", 1)
        parsed.append((piece_id.strip(), label.strip()))

    return parsed


def group_parts_in_annotation_order(row):
    """
    Groups repeated labels in annotation order.

    Example:
      3: base; 2: base; 6: base; 7: backrest

    becomes:
      base -> pieces 3, 2, 6
      backrest -> piece 7
    """
    part_json = parse_json_cell(row.get("part_annotations_json", ""))
    color_json = parse_json_cell(row.get("piece_to_color_name_json", ""))

    ordered_piece_labels = parse_part_annotations_order(row.get("part_annotations", ""))

    if not ordered_piece_labels and part_json:
        ordered_piece_labels = list(part_json.items())

    grouped = []
    label_to_group = {}

    for piece_id, label in ordered_piece_labels:
        piece_id = str(piece_id)
        label = clean_text(label)

        if not label:
            continue

        if label not in label_to_group:
            group = {
                "label": label,
                "pieces": [],
                "colors": [],
                "shapes": [],
            }
            grouped.append(group)
            label_to_group[label] = group

        group = label_to_group[label]
        group["pieces"].append(piece_id)

        color = clean_text(color_json.get(piece_id, ""))
        shape = clean_text(PIECE_TO_SHAPE.get(piece_id, ""))

        if color:
            group["colors"].append(color)

        if shape:
            group["shapes"].append(shape)

    for i, group in enumerate(grouped, start=1):
        unique_colors = []
        for color in group["colors"]:
            if color not in unique_colors:
                unique_colors.append(color)

        group["part_index"] = i
        group["unique_colors"] = unique_colors
        group["color"] = unique_colors[0] if len(unique_colors) == 1 else " and ".join(unique_colors)
        group["single_piece"] = len(group["pieces"]) == 1
        group["single_color"] = len(unique_colors) == 1

        flag_col = f"part_description_shape_{i}_single_shape_color"
        group["single_shape_color_flag"] = parse_bool_cell(row.get(flag_col, False))

    return grouped


def coarse_shape(shape):
    """
    Used to avoid swaps like small triangle -> large triangle
    when you want triangle/square/parallelogram-level changes.
    """
    shape = clean_text(shape).lower()

    if "triangle" in shape:
        return "triangle"

    if "square" in shape:
        return "square"

    if "parallelogram" in shape:
        return "parallelogram"

    return shape


def shape_phrase_from_group(group):
    """
    Single-piece group:
      square + blue -> a blue square
      large triangle + red -> a large red triangle
      parallelogram + purple -> a purple parallelogram
    """
    if not group["shapes"] or not group["unique_colors"]:
        return ""

    shape = group["shapes"][0]
    color = group["unique_colors"][0]

    if "triangle" in shape:
        size = shape.replace(" triangle", "")
        return f"a {size} {color} triangle"

    return f"a {color} {shape}"


def first_sentence_part(label, whole_object, replacement_phrase):
    return f"The {label} of the {whole_object} is {replacement_phrase}."


def continuation_part(label, replacement_phrase):
    return f"the {label} is {replacement_phrase}"


def jaccard_label_similarity(a, b):
    return jaccard_similarity(a, b)


def choose_least_similar_shape_candidate(source_group, candidate_groups):
    """
    Among candidate groups, prefer a different coarse shape.
    Then choose the one with the lowest label similarity.
    """
    valid = []

    source_shape = source_group["shapes"][0] if source_group["shapes"] else ""
    source_coarse = coarse_shape(source_shape)

    for cand in candidate_groups:
        if cand["part_index"] == source_group["part_index"]:
            continue

        if not cand["shapes"]:
            continue

        cand_shape = cand["shapes"][0]
        cand_coarse = coarse_shape(cand_shape)

        # Avoid triangle -> triangle swaps, even if sizes differ.
        if cand_coarse == source_coarse:
            continue

        sim = jaccard_label_similarity(source_group["label"], cand["label"])
        valid.append((cand, sim))

    if not valid:
        return None

    valid = sorted(valid, key=lambda x: x[1])
    return valid[0][0]


def add_within_picture_swaps_for_row(row, max_swaps=3):
    """
    Creates:
      same_picture_1_shape_swap_1..3
      loose_swap_item_1..3

    same_picture_1_shape_swap:
      Uses only parts that are marked single_shape_color=True.
      Swaps a source part into the shape/color phrase of the least-similar
      candidate with a different coarse shape.

    loose_swap_item:
      For non-single-shape-color parts, replaces them with any available
      single-piece item phrase from the same picture.
    """
    whole_object = without_leading_article(row.get("whole_annotation", ""))
    groups = group_parts_in_annotation_order(row)

    output = {}

    for i in range(1, max_swaps + 1):
        output[f"same_picture_1_shape_swap_{i}"] = ""
        output[f"same_picture_1_shape_swap_{i}_source_part"] = ""
        output[f"same_picture_1_shape_swap_{i}_replacement_part"] = ""
        output[f"same_picture_1_shape_swap_{i}_source_shape"] = ""
        output[f"same_picture_1_shape_swap_{i}_replacement_shape"] = ""

        output[f"loose_swap_item_{i}"] = ""
        output[f"loose_swap_item_{i}_source_part"] = ""
        output[f"loose_swap_item_{i}_replacement_part"] = ""
        output[f"loose_swap_item_{i}_source_shape"] = ""
        output[f"loose_swap_item_{i}_replacement_shape"] = ""

    # --------------------------------------------------------
    # 1. same_picture_1_shape_swap
    # --------------------------------------------------------

    single_shape_groups = [
        g for g in groups
        if g["single_shape_color_flag"]
        and g["single_piece"]
        and g["single_color"]
        and shape_phrase_from_group(g)
    ]

    same_picture_swaps = []

    if len(single_shape_groups) >= 2:
        for source in single_shape_groups:
            replacement = choose_least_similar_shape_candidate(
                source,
                single_shape_groups,
            )

            if replacement is None:
                continue

            replacement_phrase = shape_phrase_from_group(replacement)

            swap_sentence = first_sentence_part(
                label=source["label"],
                whole_object=whole_object,
                replacement_phrase=replacement_phrase,
            )

            same_picture_swaps.append({
                "sentence": swap_sentence,
                "source_part": source["label"],
                "replacement_part": replacement["label"],
                "source_shape": source["shapes"][0],
                "replacement_shape": replacement["shapes"][0],
            })

    # De-duplicate sentences
    seen = set()
    deduped = []
    for item in same_picture_swaps:
        key = item["sentence"]
        if key not in seen:
            seen.add(key)
            deduped.append(item)

    same_picture_swaps = deduped[:max_swaps]

    for i, item in enumerate(same_picture_swaps, start=1):
        output[f"same_picture_1_shape_swap_{i}"] = item["sentence"]
        output[f"same_picture_1_shape_swap_{i}_source_part"] = item["source_part"]
        output[f"same_picture_1_shape_swap_{i}_replacement_part"] = item["replacement_part"]
        output[f"same_picture_1_shape_swap_{i}_source_shape"] = item["source_shape"]
        output[f"same_picture_1_shape_swap_{i}_replacement_shape"] = item["replacement_shape"]

    # --------------------------------------------------------
    # 2. loose_swap_item
    # --------------------------------------------------------

    nonsingle_groups = [
        g for g in groups
        if not g["single_shape_color_flag"]
    ]

    replacement_candidates = [
        g for g in groups
        if g["single_shape_color_flag"]
        and g["single_piece"]
        and g["single_color"]
        and shape_phrase_from_group(g)
    ]

    loose_swaps = []

    for source in nonsingle_groups:
        # Prefer a replacement with a different coarse shape from all shapes in source.
        source_coarse_shapes = {coarse_shape(s) for s in source["shapes"]}

        candidates = []

        for cand in replacement_candidates:
            if cand["part_index"] == source["part_index"]:
                continue

            cand_shape = cand["shapes"][0]
            cand_coarse = coarse_shape(cand_shape)

            # Different broad shape if possible
            different_shape = cand_coarse not in source_coarse_shapes

            sim = jaccard_label_similarity(source["label"], cand["label"])

            candidates.append((cand, different_shape, sim))

        if not candidates:
            continue

        # Prefer different-shape candidates, then least-similar labels.
        candidates = sorted(candidates, key=lambda x: (not x[1], x[2]))
        replacement = candidates[0][0]

        replacement_phrase = shape_phrase_from_group(replacement)

        swap_sentence = first_sentence_part(
            label=source["label"],
            whole_object=whole_object,
            replacement_phrase=replacement_phrase,
        )

        loose_swaps.append({
            "sentence": swap_sentence,
            "source_part": source["label"],
            "replacement_part": replacement["label"],
            "source_shape": "; ".join(source["shapes"]),
            "replacement_shape": replacement["shapes"][0],
        })

    # De-duplicate sentences
    seen = set()
    deduped = []
    for item in loose_swaps:
        key = item["sentence"]
        if key not in seen:
            seen.add(key)
            deduped.append(item)

    loose_swaps = deduped[:max_swaps]

    for i, item in enumerate(loose_swaps, start=1):
        output[f"loose_swap_item_{i}"] = item["sentence"]
        output[f"loose_swap_item_{i}_source_part"] = item["source_part"]
        output[f"loose_swap_item_{i}_replacement_part"] = item["replacement_part"]
        output[f"loose_swap_item_{i}_source_shape"] = item["source_shape"]
        output[f"loose_swap_item_{i}_replacement_shape"] = item["replacement_shape"]

    return pd.Series(output)


# ============================================================
# Run caption swaps for all requested text columns
# ============================================================

print("Loading Sentence-BERT model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

COLUMN_PREFIXES = {
    "photo_description": "photo_description",
    "part_description": "part_description",
    "part_description_shape": "part_description_shape",
}

for text_col, prefix in COLUMN_PREFIXES.items():
    df = add_similarity_swaps_for_column(
        df=df,
        text_col=text_col,
        prefix=prefix,
        sbert_model=model,
    )


# ============================================================
# Run within-picture shape swaps
# ============================================================

print("\nCreating within-picture shape swaps...")

swap_cols = df.apply(
    lambda row: add_within_picture_swaps_for_row(row, max_swaps=MAX_SWAP_COLUMNS),
    axis=1,
)

df = pd.concat([df, swap_cols], axis=1)


# ============================================================
# Save
# ============================================================

df.to_csv(output_path, index=False)

print(f"\nSaved to {output_path}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

preview_cols = [
    "photo_description",
    "random_photo_description",
    "most_similar_photo_description_sbert",
    "least_similar_photo_description_sbert",
    "part_description",
    "random_part_description",
    "most_similar_part_description_sbert",
    "least_similar_part_description_sbert",
    "part_description_shape",
    "same_picture_1_shape_swap_1",
    "same_picture_1_shape_swap_2",
    "same_picture_1_shape_swap_3",
    "loose_swap_item_1",
    "loose_swap_item_2",
    "loose_swap_item_3",
]

preview_cols = [c for c in preview_cols if c in df.columns]
print(df[preview_cols].head())